# 面向长时运行智能体的上下文编辑与记忆

跨多个会话运行或处理长时任务的 AI 智能体面临两个关键挑战：对话之间会丢失已学到的模式，并且在长时间交互过程中上下文窗口会被填满。

本教程演示如何利用 Claude 的记忆工具和上下文编辑能力来应对这些挑战。

## 目录

1. [引言：为什么记忆很重要](#introduction)
2. [使用场景](#use-cases)
3. [快速上手示例](#quick-start)
4. [工作原理](#how-it-works)
5. [代码审查助手演示](#demo)
6. [真实世界应用](#real-world)
7. [最佳实践](#best-practices)

## 前置条件

**必备知识：**
- Python 基础（函数、类、async/await 基础）
- 对 REST API 和 JSON 的基本理解

**必备工具：**
- Python 3.10 或更高版本
- Anthropic API 密钥（[在此获取](https://console.anthropic.com/)）

**建议具备：**
- 熟悉并发编程概念（线程、async）
- 对 LLM 上下文窗口的基本理解

## 设置

### VSCode 用户

```bash
# 1. Create virtual environment
python -m venv .venv

# 2. Activate it
source .venv/bin/activate  # macOS/Linux
# or: .venv\Scripts\activate  # Windows

# 3. Install dependencies
pip install -r requirements.txt

# 4. In VSCode: Select .venv as kernel (top right)
```

### API 密钥

```bash
cp .env.example .env
# Edit .env and add your ANTHROPIC_API_KEY
```

从以下地址获取你的 API 密钥：https://console.anthropic.com/

## 1. 引言：为什么记忆很重要 {#introduction}

本教程演示了[面向 AI 智能体的有效上下文工程](https://www.anthropic.com/engineering/effective-context-engineering-for-ai-agents)一文中所述上下文工程模式的实际实现。那篇文章解释了为什么上下文是有限资源、注意力预算如何运作，以及构建高效智能体的策略——这些技术将在这里付诸实践。

### 问题

大语言模型的上下文窗口是有限的（Claude 4 系列模型为 200k 词元）。这个数字看似很大，但会带来若干挑战：

- **上下文限制**：长对话或复杂任务可能超出可用的上下文
- **计算成本**：处理大上下文的代价很高——注意力机制的开销随规模呈二次增长
- **重复模式**：跨对话的相似任务每次都需要重新解释上下文
- **信息丢失**：上下文被填满后，早期的重要信息会丢失

### 解决方案

Claude 4 模型引入了强大的上下文管理能力：

1. **记忆工具**（`memory_20250818`）：实现跨对话学习
   - Claude 可以把它学到的内容写下来，供将来参考
   - 基于文件的系统，位于 `/memories` 目录下
   - 客户端实现让你拥有完全的控制权

2. **上下文编辑**：以两种策略自动管理上下文：
   - **工具使用清理**（`clear_tool_uses_20250919`）：上下文变大时清除旧的工具结果
   - **思考管理**（`clear_thinking_20251015`）：管理扩展思考块（需要启用思考）
   - 可配置的触发条件和保留策略

### 收益

构建能**在你的特定任务上随时间越做越好**的 AI 智能体：

- **会话 1**：Claude 解决问题，并把模式写下来
- **会话 2**：Claude 立即应用已学到的模式（更快！）
- **长会话**：上下文编辑让对话保持在可控范围内

可以把它想象成给 Claude 一个笔记本，让它记笔记并回头查阅——就像人类一样。

### 你将学到什么

完成本教程后，你将能够：
- **实现**用于跨对话学习的记忆工具
- **配置**上下文编辑以管理长时运行的会话
- **应用**记忆安全与组织的最佳实践

## 1. 引言：为什么记忆很重要 {#introduction}

本教程演示了[面向 AI 智能体的有效上下文工程](https://www.anthropic.com/engineering/effective-context-engineering-for-ai-agents)一文中所述上下文工程模式的实际实现。那篇文章解释了为什么上下文是有限资源、注意力预算如何运作，以及构建高效智能体的策略——这些技术将在这里付诸实践。

### 问题

大语言模型的上下文窗口是有限的（Claude 4 为 200k 词元）。这个数字看似很大，但会带来若干挑战：

- **上下文限制**：长对话或复杂任务可能超出可用的上下文
- **计算成本**：处理大上下文的代价很高——注意力机制的开销随规模呈二次增长
- **重复模式**：跨对话的相似任务每次都需要重新解释上下文
- **信息丢失**：上下文被填满后，早期的重要信息会丢失

### 解决方案

Claude Sonnet 4.6 引入了两项强大能力：

1. **记忆工具**（`memory_20250818`）：实现跨对话学习
   - Claude 可以把它学到的内容写下来，供将来参考
   - 基于文件的系统，位于 `/memories` 目录下
   - 客户端实现让你拥有完全的控制权

**支持的模型**：Claude Opus 4.1（`claude-opus-4-1`）、Claude Opus 4（`claude-opus-4`）、Claude Sonnet 4.6（`claude-sonnet-4-6`）、Claude Sonnet 4（`claude-sonnet-4`）以及 Claude Haiku 4.5（`claude-haiku-4-5`）

### 收益

构建能**在你的特定任务上随时间越做越好**的 AI 智能体：

- **会话 1**：Claude 解决问题，并把模式写下来
- **会话 2**：Claude 立即应用已学到的模式（更快！）
- **长会话**：上下文编辑让对话保持在可控范围内

可以把它想象成给 Claude 一个笔记本，让它记笔记并回头查阅——就像人类一样。

## 2. 使用场景 {#use-cases}

记忆与上下文管理催生了强大的新工作流：

### 🔍 代码审查助手
- 从过往审查中学习调试模式
- 在后续会话中立即识别相似的缺陷
- 逐步积累团队特有的代码质量知识
- **可用于生产**：与 [claude-code-action](https://github.com/anthropics/claude-code-action) 集成，用于 GitHub PR 审查

### 📚 研究助手
- 跨多个会话积累某个主题的知识
- 在不同研究线索之间建立洞见关联
- 维护参考书目与来源追踪

### 💬 客户支持机器人
- 学习用户偏好与沟通风格
- 记住常见问题及其解决方案
- 从交互中构建产品知识库

### 📊 数据分析助手
- 记住数据集的模式与异常
- 存储行之有效的分析技术
- 随时间积累特定领域的洞见

**支持的模型**：Claude Opus 4.1（`claude-opus-4-1`）与 Claude Sonnet 4.6（`claude-sonnet-4-6`）

**本教程以代码审查助手为重点**，因为它清晰地展示了记忆（学习模式）与上下文编辑（处理长审查）两项能力。

## 3. 快速上手示例 {#quick-start}

让我们通过简单示例看看记忆与上下文管理的实际运作。

### 设置

首先，安装依赖并配置你的环境：

In [1]:
%%capture
# Install required packages
# Option 1: From requirements.txt
# %pip install -q -r requirements.txt

# Option 2: Direct install
%pip install -q anthropic python-dotenv ipykernel

**⚠️ 重要**：在此目录中创建 `.env` 文件：

```bash
# Copy .env.example to .env and add your API key
cp .env.example .env
```

然后编辑 `.env`，加入你在 https://console.anthropic.com/ 获取的 Anthropic API 密钥

In [2]:
import os
from typing import cast

from anthropic import Anthropic
from dotenv import load_dotenv

# Load environment variables
load_dotenv()

# Model configuration - use alias for automatic updates
MODEL = "claude-sonnet-4-6"  # Can override via ANTHROPIC_MODEL env var
if os.getenv("ANTHROPIC_MODEL"):
    MODEL = os.getenv("ANTHROPIC_MODEL")

if not API_KEY:
    raise ValueError("ANTHROPIC_API_KEY not found. Copy .env.example to .env and add your API key.")

if not MODEL:
    raise ValueError("ANTHROPIC_MODEL not found. Copy .env.example to .env and set the model.")

MODEL = cast(str, MODEL)

client = Anthropic(api_key=API_KEY)

print("✓ API key loaded")
print(f"✓ Using model: {MODEL}")

✓ API key loaded
✓ Using model: claude-sonnet-4-6


### 示例 1：记忆的基本用法

我们来看看 Claude 如何使用记忆存储信息，供将来参考。

**辅助函数**

这些示例使用 `demo_helpers.py` 中的辅助函数：

- **`run_conversation_loop()`**：处理 API 对话循环
  - 启用记忆工具后调用 Claude 的 API
  - 执行工具使用操作（记忆操作）
  - 持续进行，直到 Claude 不再使用工具
  - 返回最终响应

- **`run_conversation_turn()`**：单轮对话（用于示例 3）
  - 与上述相同，但在一次 API 调用后返回
  - 在需要细粒度控制时很有用

- **`print_context_management_info()`**：显示上下文清理统计
  - 展示节省的词元数、被清除的工具使用次数
  - 帮助直观了解上下文编辑何时触发

**⚠️ 关于清除记忆的说明**

接下来的单元格会清除所有记忆文件，为本次演示提供干净的环境。这便于多次运行笔记本以获得一致的结果。

**在生产应用中**，你应当慎重考虑是否清除全部记忆，因为这会永久移除已学到的模式。可以考虑使用选择性删除，或把记忆组织到按项目划分的目录中。

In [3]:
# Import helper functions
from memory_demo.demo_helpers import (
    run_conversation_loop,
    run_conversation_turn,
    print_context_management_info,
)
from memory_tool import MemoryToolHandler

# Initialize
client = Anthropic()
memory = MemoryToolHandler(base_path="./demo_memory")

# Clear any existing memories to start fresh
print("🧹 Clearing previous memories...")
memory.clear_all_memory()
print("✓ Memory cleared\n")

# Load example code with a race condition bug
with open("memory_demo/sample_code/web_scraper_v1.py", "r") as f:
    code_to_review = f.read()

messages = [
    {
        "role": "user",
        "content": f"I'm reviewing a multi-threaded web scraper that sometimes returns fewer results than expected. The count is inconsistent across runs. Can you find the issue?\n\n```python\n{code_to_review}\n```",
    }
]

print("=" * 60)
print("📝 SESSION 1: Learning from a bug")
print("=" * 60)

# Run conversation loop
response = run_conversation_loop(
    client=client,
    model=MODEL,
    messages=messages,
    memory_handler=memory,
    system="You are a code reviewer.",
    max_tokens=2048,
    max_turns=5,
    verbose=True,
)

print("\n" + "=" * 60)
print("✅ Session 1 complete!")
print("=" * 60)

🧹 Clearing previous memories...
✓ Memory cleared

📝 SESSION 1: Learning from a bug

🔄 Turn 1:
💬 Claude: I'll review this code for the race condition issue. Let me first check my memory, then analyze the problem.

  🔧 Memory tool: view /memories
  ✓ Result: Directory: /memories
(empty)

🔄 Turn 2:
  🔧 Memory tool: create /memories/review.md
  ✓ Result: File created successfully at /memories/review.md

🔄 Turn 3:
💬 Claude: ## Code Review: Multi-threaded Web Scraper Race Condition

### 🔴 **Critical Issue Found: Race Condition in Shared State**

You've correctly identified the problem! The code has **race conditions** caused by multiple threads modifying shared lists without synchronization.

---

### **The Problem**

**Lines with race conditions:**
```python
self.results.append(result)  # RACE CONDITION
```

**Problem:** 
- Python's `list.append()` is **NOT thread-safe** for concurrent modifications
- Multiple threads simultaneously appending can cause:
  - **Lost updates:** One thread's ap

**发生了什么？**

1. Claude 检查了它的记忆（首次运行时为空）
2. 定位到缺陷：**竞态条件**——多个线程在没有同步的情况下修改共享状态（`self.results` 和 `self.failed_urls`）
3. 将该并发模式存入记忆，供将来参考

现在见证奇迹时刻——Claude 在**新对话**中应用这个已学到的模式：

### 示例 2：跨对话学习

开启一个全新的对话——记忆依然存在！

In [4]:
# NEW conversation (empty messages)
# Load API client code with similar concurrency issue
with open("memory_demo/sample_code/api_client_v1.py", "r") as f:
    code_to_review = f.read()

messages = [
    {"role": "user", "content": f"Review this API client code:\n\n```python\n{code_to_review}\n```"}
]

print("=" * 60)
print("🚀 SESSION 2: Applying learned pattern")
print("=" * 60)

# Run conversation loop
response = run_conversation_loop(
    client=client,
    model=MODEL,
    messages=messages,
    memory_handler=memory,
    system="You are a code reviewer.",
    max_tokens=2048,
    max_turns=5,
    verbose=True,
)

print("\n" + "=" * 60)
print("✅ Session 2 complete!")
print("=" * 60)

🚀 SESSION 2: Applying learned pattern

🔄 Turn 1:
  🔧 Memory tool: view /memories
  ✓ Result: Directory: /memories
- review.md

🔄 Turn 2:
  🔧 Memory tool: view /memories/review.md
  ✓ Result:    1: # Code Review: Multi-threaded Web Scraper
   2: 
   3: ## Issue
   4: Revi...

🔄 Turn 3:
  🔧 Memory tool: str_replace /memories/review.md
  ✓ Result: File /memories/review.md has been edited successfully

🔄 Turn 4:
💬 Claude: ## Code Review: Async API Client

### Summary
This code has **concurrency issues** related to shared mutable state being accessed from multiple coroutines without proper synchronization. While the author correctly identifies these as bugs in comments, let me provide a comprehensive review.

However, the actual race condition risk here is **somewhat overstated** because:
- List `.append()` is atomic in CPython
- The `+=` operation on integers is also atomic
- Coroutines only switch at `await` points, and there are none between the operations

**BUT** this is still problema

**注意其中的差别：**

- Claude **立即检查了记忆**，找到了线程安全/并发模式
- **瞬间**识别出异步代码中的类似问题，无需重新学习
- 响应**更快**，因为它直接应用了关于共享可变状态的已有知识

这就是**跨对话学习**的实际效果！

### 示例 3：清除上下文的同时保留记忆

在包含大量代码文件的**长审查会话**中会发生什么？

- 上下文被此前审查的工具结果填满
- 但记忆（已学到的模式）必须保留！

让我们触发**上下文编辑**，看看 Claude 如何自动处理这种情况。

**关于配置的说明：** 我们使用 `clear_at_least: 50` 词元，因为记忆工具的操作结果很小（每个约 50-150 词元）。在生产环境中，如果工具结果较大（如网页搜索或代码执行），应使用更高的值，例如 3000-5000 词元。

In [5]:
# Configure context management with BOTH clearing strategies
# Low thresholds for demo - in production use 30-40k tokens
CONTEXT_MANAGEMENT = {
    "edits": [
        # Thinking management MUST come first when combining strategies
        {
            "type": "clear_thinking_20251015",
            "keep": {"type": "thinking_turns", "value": 1}  # Keep only last turn's thinking
        },
        {
            "type": "clear_tool_uses_20250919",
            "trigger": {"type": "input_tokens", "value": 5000},  # Low threshold for demo
            "keep": {"type": "tool_uses", "value": 2},  # Keep last 2 tool uses
            "clear_at_least": {"type": "input_tokens", "value": 2000}
        }
    ]
}

# Extended thinking config (required for clear_thinking strategy)
THINKING = {
    "type": "enabled",
    "budget_tokens": 1024  # Budget for thinking per turn
}

# Continue from previous session - memory persists!
print("=" * 60)
print("📚 SESSION 3: Long review session with context clearing")
print("=" * 60)
print()

# Clean up messages - remove any empty content from previous session
# This ensures we have a valid message state to continue from
cleaned_messages = []
for msg in messages:
    if isinstance(msg.get("content"), list):
        # Filter out empty content blocks
        content = [c for c in msg["content"] if c]
        if content:
            cleaned_messages.append({"role": msg["role"], "content": content})
    elif msg.get("content"):
        cleaned_messages.append(msg)

messages = cleaned_messages

# Review 1: Data processor (larger file)
with open("memory_demo/sample_code/data_processor_v1.py", "r") as f:
    data_processor_code = f.read()

messages.append({
    "role": "user",
    "content": f"Review this data processor:\n\n```python\n{data_processor_code}\n```"
})

print("📝 Review 1: Data processor")
response = run_conversation_turn(
    client=client,
    model=MODEL,
    messages=messages,
    memory_handler=memory,
    system="You are a code reviewer.",
    context_management=CONTEXT_MANAGEMENT,
    thinking=THINKING,
    max_tokens=4096,
    verbose=True
)

# Add response to messages
messages.append({"role": "assistant", "content": response[1]})
if response[2]:
    messages.append({"role": "user", "content": response[2]})

print(f"  📊 Input tokens: {response[0].usage.input_tokens:,}")
context_cleared, saved = print_context_management_info(response[0])
print()

# Review 2: SQL code
with open("memory_demo/sample_code/sql_query_builder.py", "r") as f:
    sql_code = f.read()

messages.append({
    "role": "user",
    "content": f"Review this SQL query builder:\n\n```python\n{sql_code}\n```"
})

print("📝 Review 2: SQL query builder")
response = run_conversation_turn(
    client=client,
    model=MODEL,
    messages=messages,
    memory_handler=memory,
    system="You are a code reviewer.",
    context_management=CONTEXT_MANAGEMENT,
    thinking=THINKING,
    max_tokens=4096,
    verbose=True
)

messages.append({"role": "assistant", "content": response[1]})
if response[2]:
    messages.append({"role": "user", "content": response[2]})

print(f"  📊 Input tokens: {response[0].usage.input_tokens:,}")
context_cleared, saved = print_context_management_info(response[0])
print()

# Review 3: Add one more review to ensure we trigger clearing
with open("memory_demo/sample_code/web_scraper_v1.py", "r") as f:
    scraper_code = f.read()

messages.append({
    "role": "user",
    "content": f"Quick check - any issues here?\n\n```python\n{scraper_code}\n```"
})

print("📝 Review 3: Web scraper (should trigger clearing)")
response = run_conversation_turn(
    client=client,
    model=MODEL,
    messages=messages,
    memory_handler=memory,
    system="You are a code reviewer.",
    context_management=CONTEXT_MANAGEMENT,
    thinking=THINKING,
    max_tokens=4096,
    verbose=True
)

messages.append({"role": "assistant", "content": response[1]})
if response[2]:
    messages.append({"role": "user", "content": response[2]})

print(f"  📊 Input tokens: {response[0].usage.input_tokens:,}")
context_cleared, saved = print_context_management_info(response[0])

print()
print("=" * 60)
print("✅ Session 3 complete!")
print("=" * 60)

📚 SESSION 3: Long review session with context clearing

📝 Review 1: Data processor
🧠 Thinking: The user wants me to review a data processor with multiple concurrency and thread-safety issues. Let...
  🔧 Memory tool: str_replace /memories/review.md
  ✓ Result: File /memories/review.md has been edited successfully
  📊 Input tokens: 6,611
  ℹ️  Context below threshold - no clearing triggered

📝 Review 2: SQL query builder
🧠 Thinking: The user is now asking me to review a SQL query builder with SQL injection vulnerabilities. This is ...
  🔧 Memory tool: str_replace /memories/review.md
  ✓ Result: File /memories/review.md has been edited successfully
  📊 Input tokens: 7,923
  ✂️  Context editing triggered!
      • Cleared 1 thinking turn(s), saved 166 tokens
      • After clearing: 7,923 tokens

📝 Review 3: Web scraper (should trigger clearing)
🧠 Thinking: This is a quick check request for a web scraper with threading issues. Let me quickly identify the p...
  🔧 Memory tool: str_replace /me

**刚才发生了什么？**

在启用扩展思考的多次审查过程中，随着上下文增长，上下文编辑被自动应用：
1. **清除思考块**——先移除此前轮次留下的旧思考内容
2. **清除工具结果**——超过阈值后移除旧的记忆工具结果
3. **记忆文件完好**——Claude 仍能查询已学到的模式
4. **词元用量受控**——从思考内容和工具结果两方面节省词元

这展示了关键收益：
- **短期记忆**（对话上下文 + 思考）→ 被清除以节省空间
- **长期记忆**（存储的模式）→ 跨会话持续保留

让我们验证记忆在清除之后仍然存在：

In [6]:
# Verify memory persists after context clearing
import os

print("📂 Memory files in demo_memory/:")
print()

for root, dirs, files in os.walk("./demo_memory"):
    # Calculate relative path for display
    level = root.replace("./demo_memory", "").count(os.sep)
    indent = "  " * level
    folder_name = os.path.basename(root) or "demo_memory"
    print(f"{indent}{folder_name}/")

    sub_indent = "  " * (level + 1)
    for file in files:
        file_path = os.path.join(root, file)
        size = os.path.getsize(file_path)
        print(f"{sub_indent}├── {file} ({size} bytes)")

print()
print("✅ All learned patterns preserved despite context clearing!")

📂 Memory files in demo_memory/:

demo_memory/
  memories/
    ├── review.md (318 bytes)

✅ All learned patterns preserved despite context clearing!


## 4. 工作原理 {#how-it-works}

### 记忆工具架构

记忆工具运行在**客户端**——存储由你控制。Claude 发起工具调用，由你的应用执行。

#### 记忆工具命令

| 命令 | 说明 | 示例 |
|---------|-------------|---------|
| `view` | 显示目录或文件内容 | `{"command": "view", "path": "/memories"}` |
| `create` | 创建或覆盖文件 | `{"command": "create", "path": "/memories/notes.md", "file_text": "..."}` |
| `str_replace` | 替换文件中的文本 | `{"command": "str_replace", "path": "...", "old_str": "...", "new_str": "..."}` |
| `insert` | 在指定行号插入文本 | `{"command": "insert", "path": "...", "insert_line": 2, "insert_text": "..."}` |
| `delete` | 删除文件或目录 | `{"command": "delete", "path": "/memories/old.txt"}` |
| `rename` | 重命名或移动文件 | `{"command": "rename", "old_path": "...", "new_path": "..."}` |

完整实现（含路径校验与安全措施）见 `memory_tool.py`。

### 思考管理（`clear_thinking_20251015`）

使用扩展思考时，思考块会不断累积并消耗词元。`clear_thinking` 策略会自动管理这些思考块。

**重要**：此策略要求在 API 调用中启用 `thinking`。

**API 调用模式**（启用扩展思考）：

```python
response = client.beta.messages.create(
    betas=["context-management-2025-06-27"],  # Required beta flag
    model="claude-sonnet-4-6",
    messages=messages,
    tools=[{"type": "memory_20250818", "name": "memory"}],
    thinking={"type": "enabled", "budget_tokens": 10000},  # Enable thinking
    context_management={  # Context editing config
        "edits": [
            {
                "type": "clear_thinking_20251015",
                "keep": {"type": "thinking_turns", "value": 1}  # Keep last turn only
            },
            {
                "type": "clear_tool_uses_20250919",
                "trigger": {"type": "input_tokens", "value": 35000},
                "keep": {"type": "tool_uses", "value": 5}
            }
        ]
    },
    max_tokens=2048
)
```

**要点：**
- 组合多种策略时，`clear_thinking` 必须排在**最前**
- 需要启用扩展思考（`thinking={"type": "enabled", ...}`）
- 使用 `"keep": "all"` 保留全部思考块，以获得最大缓存命中
- 思考管理的触发条件是可选的（依据 `keep` 值进行清除）

### 理解演示代码

`code_review_demo.py` 中的关键实现细节：

```python
class CodeReviewAssistant:
    def __init__(self, memory_storage_path="./memory_storage"):
        self.client = Anthropic()
        self.memory_handler = MemoryToolHandler(base_path=memory_storage_path)
        self.messages = []
    
    def review_code(self, code, filename, description=""):
        # 1. Add user message
        self.messages.append({...})
        
        # 2. Conversation loop with tool execution
        while True:
            response = self.client.beta.messages.create(
                model=MODEL,
                system=self._create_system_prompt(),
                messages=self.messages,
                tools=[{"type": "memory_20250818", "name": "memory"}],
                betas=["context-management-2025-06-27"],
                context_management=CONTEXT_MANAGEMENT
            )
            
            # 3. Execute tool uses
            tool_results = []
            for content in response.content:
                if content.type == "tool_use":
                    result = self._execute_tool_use(content)
                    tool_results.append({...})
            
            # 4. Continue if there are tool uses, otherwise done
            if tool_results:
                self.messages.append({"role": "user", "content": tool_results})
            else:
                break
```

**关键模式**：只要存在工具使用，就持续调用 API，执行这些工具调用并把结果回传。

### Claude 究竟学到了什么

这正是记忆的强大之处——**语义级模式识别**，而不仅仅是语法：

**会话 1：基于线程的网页抓取器**

```python
# Bug: Race condition
class WebScraper:
    def __init__(self):
        self.results = []  # Shared state!
    
    def scrape_urls(self, urls):
        with ThreadPoolExecutor() as executor:
            for future in as_completed(futures):
                self.results.append(future.result())  # RACE!
```

**Claude 存入记忆的内容**（示例文件：`/memories/concurrency_patterns/thread_safety.md`）：

当 Claude 遇到这种模式时，它会把以下洞见写入记忆文件：
- **症状**：并发操作的结果不一致
- **原因**：共享可变状态（列表/字典）被多个线程修改
- **解决方案**：使用锁、线程安全的数据结构，或改为返回结果
- **危险信号**：线程回调中使用实例变量、声明了锁却未使用、计数器自增

---

**会话 2：异步 API 客户端**（全新对话！）

Claude 首先检查记忆，找到线程安全模式，然后：
1. **识别**出异步代码中的相似模式（协程同样可能交错执行）
2. **立即应用**解决方案（无需重新学习）
3. **结合已存储的知识**进行解释

```python
# Claude spots this immediately:
async def fetch_all(self, endpoints):
    for coro in asyncio.as_completed(tasks):
        self.responses.append(await coro)  # Same pattern!
```

---

**为什么这很重要：**

- ❌ **语法检查器**完全发现不了竞态条件
- ✅ **Claude 学会了**架构模式，并能跨场景应用
- ✅ **跨语言**：该模式同样适用于 Go、Java、Rust 的并发代码
- ✅ **持续进步**：每次审查都会充实知识库

### 示例代码文件

演示使用以下示例文件（都含有并发/线程安全缺陷）：

- `memory_demo/sample_code/web_scraper_v1.py`——竞态条件：线程修改共享状态
- `memory_demo/sample_code/api_client_v1.py`——异步场景下的类似并发缺陷
- `memory_demo/sample_code/data_processor_v1.py`——用于长会话演示的多个并发问题

我们来看其中一个：

**`memory_demo/sample_code/web_scraper_v1.py`**

```python
"""
Concurrent web scraper with a race condition bug.
Multiple threads modify shared state without synchronization.
"""

import time
from concurrent.futures import ThreadPoolExecutor, as_completed
from typing import List, Dict

import requests


class WebScraper:
    """Web scraper that fetches multiple URLs concurrently."""

    def __init__(self, max_workers: int = 10):
        self.max_workers = max_workers
        self.results = []  # BUG: Shared mutable state accessed by multiple threads!
        self.failed_urls = []  # BUG: Another race condition!

    def fetch_url(self, url: str) -> Dict[str, any]:
        """Fetch a single URL and return the result."""
        try:
            response = requests.get(url, timeout=5)
            response.raise_for_status()
            return {
                "url": url,
                "status": response.status_code,
                "content_length": len(response.content),
            }
        except requests.exceptions.RequestException as e:
            return {"url": url, "error": str(e)}

    def scrape_urls(self, urls: List[str]) -> List[Dict[str, any]]:
        """
        Scrape multiple URLs concurrently.

        BUG: self.results is accessed from multiple threads without locking!
        This causes race conditions where results can be lost or corrupted.
        """
        with ThreadPoolExecutor(max_workers=self.max_workers) as executor:
            futures = [executor.submit(self.fetch_url, url) for url in urls]

            for future in as_completed(futures):
                result = future.result()

                # RACE CONDITION: Multiple threads append to self.results simultaneously
                if "error" in result:
                    self.failed_urls.append(result["url"])  # RACE CONDITION
                else:
                    self.results.append(result)  # RACE CONDITION

        return self.results
```

**缺陷**：多个线程在没有加锁的情况下修改 `self.results` 和 `self.failed_urls`！

Claude 将会：
1. 识别竞态条件
2. 把该模式存入 `/memories/concurrency_patterns/thread_safety.md`
3. 在会话 2 中把这一并发模式应用到异步代码

### 演示总览

我们构建了一个完整的代码审查助手。实现位于 `memory_demo/code_review_demo.py`。

**运行交互式演示：**
```bash
python memory_demo/code_review_demo.py
```

该演示展示：
1. **会话 1**：审查含有缺陷的 Python 代码 → Claude 学到该模式
2. **会话 2**：审查相似代码（新对话）→ Claude 应用该模式
3. **会话 3**：长审查会话 → 上下文编辑使其保持可控

## 7. 最佳实践与安全 {#best-practices}

### 记忆管理

**应当做：**
- ✅ 存储与任务相关的模式，而不是对话历史
- ✅ 用清晰的目录结构进行组织
- ✅ 使用有描述性的文件名
- ✅ 定期审查并清理记忆

**不应当做：**
- ❌ 存储敏感信息（密码、API 密钥、个人身份信息）
- ❌ 让记忆无限制增长
- ❌ 不加区分地存储所有内容

### 安全：路径穿越防护

**关键**：务必校验路径，以防目录穿越攻击。实现见 `memory_tool.py`。

### 安全：记忆投毒

**⚠️ 重大风险**：记忆文件会被读回 Claude 的上下文，使其成为提示注入的潜在途径。

**缓解策略：**
1. **内容净化**：存储前过滤危险模式
2. **记忆作用域隔离**：按用户/按项目隔离  
3. **记忆审计**：记录并扫描所有记忆操作
4. **提示词工程**：指示 Claude 忽略记忆中的指令

完整的安全实现见 `memory_tool.py`，测试见 `tests/`。

## 结语

### 你完成的内容

在本教程中，你学会了：
- ✅ **实现记忆工具**，用于跨对话学习（会话 1 和 2 展示了模式识别的延续）
- ✅ **配置上下文编辑**，设置词元触发条件与保留策略（会话 3 演示了自动清除）
- ✅ **应用安全最佳实践**，包括路径校验与记忆投毒防护

### 应用这些模式

**在你的项目中：**
1. 从单个模式记忆文件起步（例如 `/memories/patterns.md`）
2. 生产环境中把上下文编辑触发阈值设为 30-40k 词元
3. 实现按项目隔离的记忆，防止交叉污染

**其他应用：**
- **客户支持**：存储用户偏好与常见问题的解决方案
- **研究助手**：跨会话积累领域知识
- **数据分析**：记住数据集的特征与行之有效的技术

### 后续步骤

- **生产部署**：使用 [claude-code-action](https://github.com/anthropics/claude-code-action) 进行 GitHub PR 审查
- **安全加固**：查阅 `memory_tool.py` 中的记忆投毒缓解措施
- **扩展思考**：探索面向计算密集型任务的思考管理

### 资源

- [记忆工具文档](https://docs.claude.com/en/docs/agents-and-tools/tool-use/memory-tool)
- [Claude API 参考](https://docs.claude.com/en/api/messages)
- [支持](https://support.claude.com)

记忆与上下文管理目前处于 **beta** 阶段。欢迎分享你的反馈，帮助我们改进！